In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import differential_evolution


DEV_CSV = "ru_calib_inf/predictions_analysis.csv"
OUTPUT_BIAS_CSV = "ru_calib_inf/class_biases.csv"

LABELS = [
    "ABBREVIATION", "ALTERNATIVE_NAME", "SUBCLASS_OF", "PART_OF",
    "TREATED_USING", "ORIGINS_FROM", "TO_DETECT_OR_STUDY", "AFFECTS",
    "HAS_CAUSE", "APPLIED_TO", "USED_IN", "ASSOCIATED_WITH",
    "PHYSIOLOGY_OF", "FINDING_OF",
    "no_relation"
]


df = pd.read_csv(DEV_CSV)
df = df[df["gold_label"].notna()].copy()

score_cols = [f"score_avg__{label}" for label in LABELS]
missing = [c for c in score_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing score columns: {missing}")

score_matrix = df[score_cols].values.astype(np.float32)

label_to_idx = {label: i for i, label in enumerate(LABELS)}
idx_to_label = {i: label for label, i in label_to_idx.items()}
gold_idx = np.array([label_to_idx[x] for x in df["gold_label"].values], dtype=np.int64)


def predict_with_biases(score_matrix, biases):
    adjusted = score_matrix + biases[None, :]
    return adjusted.argmax(axis=1)

def macro_f1_from_biases(biases):
    pred_idx = predict_with_biases(score_matrix, biases)
    return f1_score(gold_idx, pred_idx, average="macro")

def objective(biases):
    return -macro_f1_from_biases(biases)


initial_biases = np.zeros(len(LABELS), dtype=np.float32)
baseline_macro_f1 = macro_f1_from_biases(initial_biases)
print(f"Baseline macro-F1 (no bias): {baseline_macro_f1:.6f}")



bounds = []
for label in LABELS:
    # при необходмости калибровки всех классов, инициализация следующая
    # if label == "no_relation":
    #     bounds.append((0.0, 0.0))
    # else:
    #     bounds.append((-1.5, 1.5))
    if label == "no_relation":
        bounds.append((-1.5, 1.5))
    else:
        bounds.append((0.0, 0.0))

result = differential_evolution(
    objective,
    bounds=bounds,
    strategy="best1bin",
    maxiter=50,
    popsize=10,
    tol=1e-3,
    mutation=(0.5, 1.0),
    recombination=0.7,
    polish=False,
    disp=True,
    seed=42,
    workers=1,
)

best_biases = result.x
best_macro_f1 = -result.fun

print("\nOptimization finished.")
print(f"Best macro-F1: {best_macro_f1:.6f}")
print(f"Gain: {best_macro_f1 - baseline_macro_f1:+.6f}")


bias_df = pd.DataFrame({
    "label": LABELS,
    "bias": best_biases
}).sort_values("bias", ascending=False)

print("\nLearned biases:")
print(bias_df.to_string(index=False))

bias_df.to_csv(OUTPUT_BIAS_CSV, index=False, encoding="utf-8")
print(f"\nSaved biases to: {OUTPUT_BIAS_CSV}")


pred_idx = predict_with_biases(score_matrix, best_biases)
pred_labels = [idx_to_label[i] for i in pred_idx]
df["pred_label_calibrated"] = pred_labels

print("\nClassification report after calibration on DEV:")
print(classification_report(df["gold_label"], df["pred_label_calibrated"], labels=LABELS, zero_division=0))

df.to_csv("logprob/predictions_analysis_calibrated.csv", index=False, encoding="utf-8")
print("Saved calibrated DEV predictions.")


Baseline macro-F1 (no bias): 0.859546
differential_evolution step 1: f(x)= -0.851639778674365
differential_evolution step 2: f(x)= -0.8585821973961566
differential_evolution step 3: f(x)= -0.8663185001464093
differential_evolution step 4: f(x)= -0.8663185001464093
differential_evolution step 5: f(x)= -0.8700984641170657
differential_evolution step 6: f(x)= -0.8700984641170657
differential_evolution step 7: f(x)= -0.8732942875193359
differential_evolution step 8: f(x)= -0.8766461889995223
differential_evolution step 9: f(x)= -0.8766461889995223
differential_evolution step 10: f(x)= -0.8766461889995223
differential_evolution step 11: f(x)= -0.8795724485369126
differential_evolution step 12: f(x)= -0.8795724485369126
differential_evolution step 13: f(x)= -0.8799683931043359
differential_evolution step 14: f(x)= -0.8799683931043359
differential_evolution step 15: f(x)= -0.8799683931043359
differential_evolution step 16: f(x)= -0.8799683931043359
differential_evolution step 17: f(x)= -0.879

In [ ]:
import pandas as pd
import numpy as np


TEST_CSV = "ru_test_inf/predictions_analysis.csv"
BIAS_CSV = "ru_calib_inf/class_biases.csv"

OUTPUT_TEST_CSV = "ru_test_inf/predictions_analysis_calibrated.csv"
OUTPUT_TEST_TSV = "ru_test_inf/predictions_rel_calibrated.tsv"

LABELS = [
    "ABBREVIATION", "ALTERNATIVE_NAME", "SUBCLASS_OF", "PART_OF",
    "TREATED_USING", "ORIGINS_FROM", "TO_DETECT_OR_STUDY", "AFFECTS",
    "HAS_CAUSE", "APPLIED_TO", "USED_IN", "ASSOCIATED_WITH",
    "PHYSIOLOGY_OF", "FINDING_OF",
    "no_relation"
]


df = pd.read_csv(TEST_CSV)
bias_df = pd.read_csv(BIAS_CSV)

label_to_bias = dict(zip(bias_df["label"], bias_df["bias"]))

# проверка
for label in LABELS:
    if label not in label_to_bias:
        raise ValueError(f"Missing bias for label: {label}")


adjusted_score_cols = []

for label in LABELS:
    raw_col = f"score_avg__{label}"
    adj_col = f"score_adj__{label}"

    if raw_col not in df.columns:
        raise ValueError(f"Missing score column: {raw_col}")

    df[adj_col] = df[raw_col] + label_to_bias[label]
    adjusted_score_cols.append(adj_col)


adj_matrix = df[adjusted_score_cols].values
best_idx = adj_matrix.argmax(axis=1)

idx_to_label = {i: label for i, label in enumerate(LABELS)}
df["pred_label_calibrated"] = [idx_to_label[i] for i in best_idx]

# margin
sorted_scores = np.sort(adj_matrix, axis=1)
df["confidence_margin_calibrated"] = sorted_scores[:, -1] - sorted_scores[:, -2]


df.to_csv(OUTPUT_TEST_CSV, index=False, encoding="utf-8")
print(f"Saved calibrated test CSV: {OUTPUT_TEST_CSV}")


required_cols = [
    "document_id",
    "head_text", "head_span", "head_type",
    "tail_text", "tail_span", "tail_type",
    "pred_label_calibrated"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns needed for TSV export: {missing}")

tsv_df = pd.DataFrame({
    "document_id": df["document_id"],
    "relation": df["pred_label_calibrated"],
    "head_text": df["head_text"],
    "head_span": df["head_span"],
    "head_type": df["head_type"],
    "tail_text": df["tail_text"],
    "tail_span": df["tail_span"],
    "tail_type": df["tail_type"],
})

tsv_df.to_csv(OUTPUT_TEST_TSV, sep="\t", index=False, encoding="utf-8")
print(f"Saved calibrated test TSV: {OUTPUT_TEST_TSV}")


Saved calibrated test CSV: ru_test_inf/predictions_analysis_calibrated.csv
Saved calibrated test TSV: ru_test_inf/predictions_rel_calibrated.tsv


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report


DEV_CSV = "dev_blind/predictions_analysis_with_gold.csv"
OUTPUT_PARAMS_JSON = "dev_blind/no_rel_bias_margin_params.json"
OUTPUT_CALIBRATED_CSV = "dev_blind/predictions_analysis_calibrated.csv"

LABELS = [
    "ABBREVIATION", "ALTERNATIVE_NAME", "SUBCLASS_OF", "PART_OF",
    "TREATED_USING", "ORIGINS_FROM", "TO_DETECT_OR_STUDY", "AFFECTS",
    "HAS_CAUSE", "APPLIED_TO", "USED_IN", "ASSOCIATED_WITH",
    "PHYSIOLOGY_OF", "FINDING_OF",
    "no_relation"
]

NO_RELATION_LABEL = "no_relation"

NO_REL_BIAS_GRID = np.arange(-1.5, 1.51, 0.05)
MARGIN_GRID = np.arange(0.0, 1.01, 0.02)


df = pd.read_csv(DEV_CSV)
df = df[df["gold_label"].notna()].copy()

score_cols = [f"score_avg__{label}" for label in LABELS]
missing = [c for c in score_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing score columns: {missing}")

score_matrix = df[score_cols].values.astype(np.float32)

label_to_idx = {label: i for i, label in enumerate(LABELS)}
idx_to_label = {i: label for label, i in label_to_idx.items()}
no_rel_idx = label_to_idx[NO_RELATION_LABEL]

gold_idx = np.array([label_to_idx[x] for x in df["gold_label"].values], dtype=np.int64)

print(f"Loaded {len(df)} examples")
print(f"Labels: {LABELS}")
print(f"no_relation index: {no_rel_idx}")


def apply_no_relation_bias_and_margin(score_matrix, no_rel_bias, margin_threshold):
    """
    1) Добавляем bias только к no_relation
    2) Считаем top1 и top2
    3) Если margin < threshold, ставим no_relation
    """
    adjusted = score_matrix.copy()
    adjusted[:, no_rel_idx] += no_rel_bias

    # top1
    pred_idx = adjusted.argmax(axis=1)

    # top2 margin
    sorted_scores = np.sort(adjusted, axis=1)
    top1_scores = sorted_scores[:, -1]
    top2_scores = sorted_scores[:, -2]
    margins = top1_scores - top2_scores

    # override to no_relation if uncertain
    pred_idx = pred_idx.copy()
    pred_idx[margins < margin_threshold] = no_rel_idx

    return pred_idx, margins, adjusted


def macro_f1_with_params(score_matrix, gold_idx, no_rel_bias, margin_threshold):
    pred_idx, _, _ = apply_no_relation_bias_and_margin(
        score_matrix, no_rel_bias, margin_threshold
    )
    return f1_score(gold_idx, pred_idx, average="macro")


baseline_pred = score_matrix.argmax(axis=1)
baseline_macro_f1 = f1_score(gold_idx, baseline_pred, average="macro")
baseline_micro_f1 = f1_score(gold_idx, baseline_pred, average="micro")

print(f"Baseline macro-F1: {baseline_macro_f1:.6f}")
print(f"Baseline micro-F1: {baseline_micro_f1:.6f}")


best_macro_f1 = -1.0
best_params = None

results = []

for no_rel_bias in NO_REL_BIAS_GRID:
    for margin_threshold in MARGIN_GRID:
        macro_f1 = macro_f1_with_params(
            score_matrix=score_matrix,
            gold_idx=gold_idx,
            no_rel_bias=no_rel_bias,
            margin_threshold=margin_threshold,
        )

        results.append({
            "no_rel_bias": float(no_rel_bias),
            "margin_threshold": float(margin_threshold),
            "macro_f1": float(macro_f1),
        })

        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_params = {
                "no_rel_bias": float(no_rel_bias),
                "margin_threshold": float(margin_threshold),
            }

print("\nBest params found:")
print(best_params)
print(f"Best macro-F1: {best_macro_f1:.6f}")
print(f"Gain over baseline: {best_macro_f1 - baseline_macro_f1:+.6f}")


best_pred_idx, best_margins, adjusted_scores = apply_no_relation_bias_and_margin(
    score_matrix=score_matrix,
    no_rel_bias=best_params["no_rel_bias"],
    margin_threshold=best_params["margin_threshold"],
)

best_pred_labels = [idx_to_label[i] for i in best_pred_idx]
df["pred_label_calibrated"] = best_pred_labels
df["confidence_margin_calibrated"] = best_margins

for i, label in enumerate(LABELS):
    df[f"score_adj__{label}"] = adjusted_scores[:, i]

macro_f1 = f1_score(gold_idx, best_pred_idx, average="macro")
micro_f1 = f1_score(gold_idx, best_pred_idx, average="micro")

print("\nClassification report after calibration:")
print(classification_report(df["gold_label"], df["pred_label_calibrated"], labels=LABELS, zero_division=0))
print(f"Macro-F1: {macro_f1:.6f}")
print(f"Micro-F1: {micro_f1:.6f}")


df.to_csv(OUTPUT_CALIBRATED_CSV, index=False, encoding="utf-8")

with open(OUTPUT_PARAMS_JSON, "w", encoding="utf-8") as f:
    json.dump(
        {
            "labels": LABELS,
            "no_relation_label": NO_RELATION_LABEL,
            "best_params": best_params,
            "baseline_macro_f1": float(baseline_macro_f1),
            "baseline_micro_f1": float(baseline_micro_f1),
            "best_macro_f1": float(macro_f1),
            "best_micro_f1": float(micro_f1),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame(results).to_csv(
    "dev_blind/no_rel_bias_margin_grid_results.csv",
    index=False,
    encoding="utf-8"
)

print(f"\nSaved calibrated predictions to: {OUTPUT_CALIBRATED_CSV}")
print(f"Saved best params to: {OUTPUT_PARAMS_JSON}")
print("Saved full grid results to: dev_blind/no_rel_bias_margin_grid_results.csv")


Loaded 28272 examples
Labels: ['ABBREVIATION', 'ALTERNATIVE_NAME', 'SUBCLASS_OF', 'PART_OF', 'TREATED_USING', 'ORIGINS_FROM', 'TO_DETECT_OR_STUDY', 'AFFECTS', 'HAS_CAUSE', 'APPLIED_TO', 'USED_IN', 'ASSOCIATED_WITH', 'PHYSIOLOGY_OF', 'FINDING_OF', 'no_relation']
no_relation index: 14
Baseline macro-F1: 0.471874
Baseline micro-F1: 0.813455

Best params found:
{'no_rel_bias': 1.5000000000000027, 'margin_threshold': 0.2}
Best macro-F1: 0.559887
Gain over baseline: +0.088013

Classification report after calibration:
                    precision    recall  f1-score   support

      ABBREVIATION       0.52      0.92      0.67        72
  ALTERNATIVE_NAME       0.32      0.38      0.35        39
       SUBCLASS_OF       0.56      0.87      0.68       744
           PART_OF       0.45      0.69      0.54       252
     TREATED_USING       0.44      0.67      0.53        81
      ORIGINS_FROM       0.37      0.81      0.51        73
TO_DETECT_OR_STUDY       0.29      0.61      0.39       113
  

In [ ]:
import json
import numpy as np
import pandas as pd


TEST_CSV = "ru_test_inf/predictions_analysis.csv"
PARAMS_JSON = "ru_calib_blind_inf/no_rel_bias_margin_params.json"

OUTPUT_TEST_CSV = "ru_test_inf/predictions_analysis_calibrated.csv"
OUTPUT_TEST_TSV = "ru_test_inf/predictions_rel_calibrated.tsv"

df = pd.read_csv(TEST_CSV)

with open(PARAMS_JSON, "r", encoding="utf-8") as f:
    params = json.load(f)

LABELS = params["labels"]
NO_RELATION_LABEL = params["no_relation_label"]
NO_REL_BIAS = params["best_params"]["no_rel_bias"]
MARGIN_THRESHOLD = params["best_params"]["margin_threshold"]

print("Loaded params:")
print("  no_rel_bias:", NO_REL_BIAS)
print("  margin_threshold:", MARGIN_THRESHOLD)

score_cols = [f"score_avg__{label}" for label in LABELS]
missing = [c for c in score_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing score columns in test CSV: {missing}")

score_matrix = df[score_cols].values.astype(np.float32)

label_to_idx = {label: i for i, label in enumerate(LABELS)}
idx_to_label = {i: label for label, i in label_to_idx.items()}
no_rel_idx = label_to_idx[NO_RELATION_LABEL]

adjusted = score_matrix.copy()
adjusted[:, no_rel_idx] += NO_REL_BIAS

pred_idx = adjusted.argmax(axis=1)

sorted_scores = np.sort(adjusted, axis=1)
top1_scores = sorted_scores[:, -1]
top2_scores = sorted_scores[:, -2]
margins = top1_scores - top2_scores

pred_idx = pred_idx.copy()
pred_idx[margins < MARGIN_THRESHOLD] = no_rel_idx

pred_labels = [idx_to_label[i] for i in pred_idx]

df["pred_label_calibrated"] = pred_labels
df["confidence_margin_calibrated"] = margins

for i, label in enumerate(LABELS):
    df[f"score_adj__{label}"] = adjusted[:, i]


df.to_csv(OUTPUT_TEST_CSV, index=False, encoding="utf-8")
print(f"Saved calibrated test analysis to: {OUTPUT_TEST_CSV}")


required_cols = [
    "document_id",
    "head_text", "head_span", "head_type",
    "tail_text", "tail_span", "tail_type"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for TSV export: {missing_cols}")

tsv_df = pd.DataFrame({
    "document_id": df["document_id"],
    "relation": df["pred_label_calibrated"],
    "head_text": df["head_text"],
    "head_span": df["head_span"],
    "head_type": df["head_type"],
    "tail_text": df["tail_text"],
    "tail_span": df["tail_span"],
    "tail_type": df["tail_type"],
})

tsv_df.to_csv(OUTPUT_TEST_TSV, sep="\t", index=False, encoding="utf-8")
tsv_df[tsv_df['relation'] != 'no_relation'].to_csv(OUTPUT_TEST_TSV, sep="\t", index=False, encoding="utf-8")
print(f"Saved calibrated submission TSV to: {OUTPUT_TEST_TSV}")


Loaded params:
  no_rel_bias: 1.5000000000000027
  margin_threshold: 0.38
Saved calibrated test analysis to: ru_test_inf/predictions_analysis_calibrated.csv
Saved calibrated submission TSV to: ru_test_inf/predictions_rel_calibrated.tsv
